# Sign Language Multiclass Classification with PyTorch

This notebook adapts the course section **Multiclass Classification with PyTorch (Addendum)** to our project dataset. The structure is intentionally close to the addendum: dataset, dataloaders, network, cross-entropy loss, minibatch training, accuracy, and model saving/loading.


In [ ]:
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

from Dataset import SignLanguageDataset
from model import LinearNet, Net

random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Load labels and inspect the dataset

The label file has no header. Each row contains the class label and the image filename.

In [ ]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "sign_lang_train" / "sign_lang_train"
LABELS_FILE = DATA_DIR / "labels.csv"

assert DATA_DIR.exists(), f"Dataset directory not found: {DATA_DIR.resolve()}"
assert LABELS_FILE.exists(), f"Labels file not found: {LABELS_FILE.resolve()}"

df = pd.read_csv(LABELS_FILE, header=None, names=["label", "image_path"])
df["label"] = df["label"].astype(str)

print(df.head())
print("Number of images:", len(df))
print("Number of classes:", df["label"].nunique())

In [ ]:
labels = sorted(df["label"].unique().tolist())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}

print(labels)
print(label_to_idx)

counts = df["label"].value_counts().sort_index()
print(counts)

In [ ]:
plt.figure(figsize=(12, 4))
counts.plot(kind="bar")
plt.title("Class distribution")
plt.xlabel("Class")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

## 2. Hyperparameters

Compared to the MNIST addendum, the important adaptations are `INPUT_SIZE = 64 * 64` and `OUTPUT_SIZE = 36`.

In [ ]:
# Hyperparameters adapted from the multiclass PyTorch addendum
LEARNING_RATE = 0.001
MOMENTUM = 0.9
NUM_EPOCHS = 15
HIDDEN_SIZE = 256
TRAIN_BATCH_SIZE = 64
TEST_BATCH_SIZE = 64

IMAGE_SIZE = 64
INPUT_SIZE = IMAGE_SIZE * IMAGE_SIZE  # 64x64 grayscale images -> 4096 features
OUTPUT_SIZE = len(labels)             # 36 classes

print("INPUT_SIZE:", INPUT_SIZE)
print("OUTPUT_SIZE:", OUTPUT_SIZE)

## 3. Transforms

The MLP receives flattened tensors. The images are resized, converted to tensors, and normalized. Light random rotation is only used for the training set.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

## 4. Stratified train/validation/test split

The classes are imbalanced, so we use a stratified split instead of a purely random split.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=random_seed,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=random_seed,
    stratify=temp_df["label"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

## 5. Dataset objects and DataLoaders

This corresponds to the DataLoader part of the course addendum. The only difference is that we use our own dataset class instead of `datasets.MNIST`.

In [ ]:
train_dataset = SignLanguageDataset(train_df, DATA_DIR, transform=train_transform, label_to_idx=label_to_idx)
val_dataset = SignLanguageDataset(val_df, DATA_DIR, transform=eval_transform, label_to_idx=label_to_idx)
test_dataset = SignLanguageDataset(test_df, DATA_DIR, transform=eval_transform, label_to_idx=label_to_idx)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    drop_last=False
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=TEST_BATCH_SIZE,
    shuffle=False,
    drop_last=False
)

num_train_batches = len(train_loader)
num_val_batches = len(val_loader)
num_test_batches = len(test_loader)

print(f"num_train_batches: {num_train_batches}")
print(f"num_val_batches: {num_val_batches}")
print(f"num_test_batches: {num_test_batches}")

In [ ]:
# View sample data
fig, ax = plt.subplots(1, 5, figsize=(10, 2))
for i in range(5):
    image, label_idx = train_dataset[i]
    ax[i].imshow(image.squeeze(0), cmap="gray")
    ax[i].set_title(f"Label: {idx_to_label[int(label_idx)]}")
    ax[i].axis("off")
plt.tight_layout()
plt.show()

## 6. Training and evaluation functions

These functions follow the same logic as the addendum: zero gradients, forward pass, loss, backward pass, optimizer step.

In [ ]:
def train_one_epoch(net, data_loader, optimizer, criterion, device):
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for data, target in data_loader:
        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = net(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += (pred == target).sum().item()
        total += target.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total
    return epoch_loss, epoch_accuracy


def evaluate(net, data_loader, criterion, device):
    net.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_targets = []
    all_predictions = []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            output = net(data)
            loss = criterion(output, target)
            pred = output.argmax(dim=1)

            running_loss += loss.item() * data.size(0)
            correct += (pred == target).sum().item()
            total += target.size(0)

            all_targets.extend(target.cpu().numpy())
            all_predictions.extend(pred.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_accuracy = correct / total
    macro_f1 = f1_score(all_targets, all_predictions, average="macro")
    return epoch_loss, epoch_accuracy, macro_f1, np.array(all_targets), np.array(all_predictions)


def train_model(net, train_loader, val_loader, optimizer, criterion, num_epochs, device):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_f1": []
    }

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(net, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1, _, _ = evaluate(net, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)
        history["val_macro_f1"].append(val_f1)

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.4f}, val_macro_f1={val_f1:.4f}"
        )

    return history

## 7. Method 1: Linear multiclass baseline

This is the simplest comparison method: one linear layer from the flattened image to 36 logits.

In [ ]:
linear_model = LinearNet(INPUT_SIZE, OUTPUT_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
linear_optimizer = optim.SGD(linear_model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

linear_history = train_model(
    linear_model,
    train_loader,
    val_loader,
    linear_optimizer,
    criterion,
    NUM_EPOCHS,
    device
)

## 8. Method 2: Multi-Layer Perceptron

This is the MLP version of the course `Net` class, adapted to 64x64 sign-language images and 36 output classes.

In [ ]:
mlp_model = Net(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE, dropout=0.3).to(device)
criterion = nn.CrossEntropyLoss()
mlp_optimizer = optim.SGD(mlp_model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)

mlp_history = train_model(
    mlp_model,
    train_loader,
    val_loader,
    mlp_optimizer,
    criterion,
    NUM_EPOCHS,
    device
)

## 9. Training curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(linear_history["val_accuracy"], label="Linear validation accuracy")
plt.plot(mlp_history["val_accuracy"], label="MLP validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Validation accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(linear_history["train_loss"], label="Linear train loss")
plt.plot(linear_history["val_loss"], label="Linear validation loss")
plt.plot(mlp_history["train_loss"], label="MLP train loss")
plt.plot(mlp_history["val_loss"], label="MLP validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and validation loss")
plt.legend()
plt.tight_layout()
plt.show()

## 10. Final test evaluation

In [ ]:
linear_test_loss, linear_test_acc, linear_test_f1, linear_targets, linear_preds = evaluate(
    linear_model, test_loader, criterion, device
)
mlp_test_loss, mlp_test_acc, mlp_test_f1, mlp_targets, mlp_preds = evaluate(
    mlp_model, test_loader, criterion, device
)

results = pd.DataFrame([
    {"model": "Linear baseline", "test_loss": linear_test_loss, "test_accuracy": linear_test_acc, "test_macro_f1": linear_test_f1},
    {"model": "MLP", "test_loss": mlp_test_loss, "test_accuracy": mlp_test_acc, "test_macro_f1": mlp_test_f1},
])

results

In [ ]:
print("MLP classification report:")
print(classification_report(mlp_targets, mlp_preds, target_names=[idx_to_label[i] for i in range(OUTPUT_SIZE)]))

In [ ]:
cm = confusion_matrix(mlp_targets, mlp_preds)
fig, ax = plt.subplots(figsize=(12, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[idx_to_label[i] for i in range(OUTPUT_SIZE)])
disp.plot(ax=ax, xticks_rotation=90, values_format="d")
plt.title("MLP confusion matrix")
plt.tight_layout()
plt.show()

## 11. Save and load the final model

The project instructions require that the model can be saved and loaded without retraining.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "saved_models"
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "mlp_sign_language.pt"

checkpoint = {
    "model_state_dict": mlp_model.state_dict(),
    "input_size": INPUT_SIZE,
    "hidden_size": HIDDEN_SIZE,
    "output_size": OUTPUT_SIZE,
    "image_size": IMAGE_SIZE,
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
}

torch.save(checkpoint, MODEL_PATH)
print("Saved model to:", MODEL_PATH)
print("Model file size in MB:", MODEL_PATH.stat().st_size / (1024 * 1024))

In [ ]:
loaded_checkpoint = torch.load(MODEL_PATH, map_location=device)
loaded_model = Net(
    loaded_checkpoint["input_size"],
    loaded_checkpoint["hidden_size"],
    loaded_checkpoint["output_size"],
    dropout=0.3
).to(device)
loaded_model.load_state_dict(loaded_checkpoint["model_state_dict"])

loaded_test_loss, loaded_test_acc, loaded_test_f1, _, _ = evaluate(
    loaded_model, test_loader, criterion, device
)

print(f"Loaded model test accuracy: {loaded_test_acc:.4f}")
print(f"Loaded model test macro F1: {loaded_test_f1:.4f}")

## What changed compared to the course addendum?

- `datasets.MNIST` was replaced by `SignLanguageDataset`.
- `INPUT_SIZE` changed from `784` to `4096` because we resize to `64x64`.
- `OUTPUT_SIZE` changed from `10` to `36`.
- We keep `nn.CrossEntropyLoss()` because this is still single-label multiclass classification.
- We keep minibatch training with `DataLoader`.
- We added a validation split, macro F1-score, confusion matrix, and model saving/loading for the project requirements.
